# Generate persistent AV inconsistency data

This notebook uses CPU/FFmpeg. It does not load FATE or train the detector. Review candidate event labels before using the full-task training configuration.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/vn-av-forensics")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/linhxm/vn-av-forensics.git", str(REPO)], check=True
    )
ROOT = REPO / "vn-av-forensics-generation"
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
DATASET = Path("/kaggle/input/YOUR_CLEAN_DATASET/dataset_v002")
assert (DATASET / "dataset_info.json").is_file()
PLAN = ROOT / "outputs/plan_two_head_v1.json"
OUTPUT = ROOT / "outputs/relations_two_head_v1"


def cli(*args):
    subprocess.run([sys.executable, "-m", "vn_av_generation", *map(str, args)], check=True)

In [ ]:
# Plan locks source/speaker splits before donors and edits.
# Existing plan is reused for resume; use new names for changed settings.
if not PLAN.exists():
    cli("plan", "--dataset", DATASET, "--output", PLAN)
print(PLAN.read_text()[:3000])

In [ ]:
cli("render", "--dataset", DATASET, "--plan", PLAN, "--output", OUTPUT)
cli("inspect", "--dataset", OUTPUT)

In [ ]:
import shutil

archive = shutil.make_archive("/kaggle/working/relations_two_head_v1", "zip", OUTPUT.parent, OUTPUT.name)
print(archive)

Download the ZIP, extract locally, run the review UI on port 8002, then finalize to manifest-reviewed.jsonl. Keep the whole dataset directory. Upload that reviewed dataset to Kaggle and run the training notebook; generation does not run again. The ZIP contains real generated videos, metadata, masks/interval labels, and review tasks.
